# M3 end-to-end pipeline (P1): from a photo or a clip to a verdict and first aid

One call, `pipeline.run_pipeline`, joins every stage built so far (plan section 2.1, rulings R82-R85):

```mermaid
flowchart LR
  A[Photo] --> B[Preprocess: RGB, exposure fix, letterbox, ImageNet norm]
  V[Video clip] --> S[video.schedule: the newest frame, at most 10 a second] -->|each kept frame| B
  B --> C{Front end: the run's F-mode}
  C -->|F0, F4, F5| D[Whole photo]
  C -->|F1| E[Detector box crop]
  C -->|F1m, F2, F3| F[Predicted mask: box crop, soft dim, hard mask]
  C -->|box_conf below tau_snake| X[No snake found: SnakeID skipped]
  D --> G[SnakeID: encoder and embedding]
  E --> G
  F --> G
  G --> H[Species head: top 3]
  G --> P[Class prototypes: best cosine against tau_unknown]
  G --> I[Genus, family and WHO heads]
  H --> J{Decision: decision.verdict}
  P --> J
  I --> J
  X --> J
  J -->|photo| R{Verdict}
  J -->|clip, each kept frame| W[4-of-6 vote, or the EMA option; no majority shows we don't know] --> R
  R -->|WHO 1 or 2 likely, viper or elapid, guarded genus| K[Dangerous: keep away]
  R -->|confident, not listed| L[Not on the WHO list, still keep away]
  R -->|no snake, far from every prototype, unsure, frames disagree| M[We don't know: keep away]
  K --> N[Response: species_card and the R0 lookup of WHO passages]
  L --> N
  M --> N
  N --> O[Screen: verdict, top 3, reason, quoted guidance with its page]
```

Each node logs a short summary of its output and its milliseconds. The top 3 come from the temperature-scaled species
head; the prototypes give the best cosine that tau_unknown reads. A clip runs the first four nodes on the frames
`video.schedule` keeps: here only the 10-frames-a-second cap skips frames, and a measured per-frame latency can be
passed instead. The screen shows what 4-of-6 voting on the per-frame verdicts gives (`video.vote_k_of_n`; the EMA
of `video.smooth_outputs` is the option): "we don't know" until 4 of the last 6 frames agree and whenever they stop
agreeing (R87), then the species card and passages of that verdict.

**This run (R84)** is local, on the CPU: 100 validation photos of the local sample (`use == "val"`, classes in turn,
seed 42) and 5 clips made from the first 5 of them by `video.make_clip` (M1 section 3.4's recipe, seeds 42-46), with
the E1 MobileViT-S run and the thresholds `decision_report` fitted on it. The front-end weights are not on this
machine and F0 needs none, so the front end is F0 with no detector: every frame is treated as showing a snake.
`RUN_DIR`, `DECISION_FILE`, `FRONTEND_DIR` and `ONNX_PATH` take the final model and front end later.
`DECISION_FILE` defaults to `RUN_DIR/decision_metrics.json` when the run folder holds one, and `load_models` refuses a
file that is not the run's own: `decision.decide` on the run's `val_outputs.npz` at the file's thresholds must give the
file's validation verdict counts.

**Outputs.** `results/m3/p1/` (committed): `photos.jsonl` and `clips.jsonl` (one line per photo or clip, with every
node's log), `examples/` (8 example outputs: 2 per verdict and 2 failures) and `summary.json`. They hold passage ids
and pages, never passage text (R6), and no image: only each photo's path, licence and attribution from the manifest.
The same 8 examples with their passage text go to `kaggle/m3_runs/p1/examples/` beside the repo, never committed.
No cell prints guideline text.

In [1]:
from pathlib import Path

WORKSPACE = Path("..").resolve().parent   # the folder holding the repo and kaggle/
RUN_DIR = Path("/Users/puneetbajaj/bluepill_local_dev/work-dirs/ds-lab-feasibility/kaggle/m3_runs/stageC/final_candidates/top1_s1/H1_top1_s1")
DECISION_FILE = Path("/Users/puneetbajaj/bluepill_local_dev/work-dirs/ds-lab-feasibility/m3-final/results/m3/final/H1_top1_s1/decision_metrics.json")
if not DECISION_FILE.exists():                      # none, so the committed copy, which load_models checks against it
    DECISION_FILE = Path("../results/m3/e1/E1_mvits/decision_metrics.json")
FRONTEND_DIR = Path("/Users/puneetbajaj/bluepill_local_dev/work-dirs/ds-lab-feasibility/kaggle/m3_runs/frontends/weights/frontends")
ONNX_PATH = None       # the run's int8 ONNX export (export.py) in place of best.pt
SAMPLE_DIR = WORKSPACE / "kaggle" / "m3_runs" / "local_sample" / "out2" / "local_sample"
N_PHOTOS = 100
N_CLIPS = 5
SEED = 42
OUT = Path("../results/m3/p1_final")
TEXT_OUT = WORKSPACE / "kaggle" / "m3_runs" / "p1_final"

In [2]:
import json
import sys
import time
from collections import Counter

SRC = Path("../src")
if SRC.is_dir() and str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

import numpy as np
import pandas as pd

from serpentguard.data import IMAGE_FOLDER, class_index
from serpentguard.decision import decide, repo_tables
from serpentguard.kb import build_kb, find_who_pdf
from serpentguard.metrics import VERDICTS
from serpentguard.pipeline import CLIP_SCOPE, FAILURE, NODES, load_models, p1_record, pick_examples, run_pipeline
from serpentguard.respond import LookupRetriever
from serpentguard.run_outputs import library_versions, subset_hash, write_json
from serpentguard.train import VAL_OUTPUTS
from serpentguard.video import load_raw_rgb, make_clip, round_robin

PDF = find_who_pdf()
assert PDF is not None, "put the guideline at refs/who_searo_2016.pdf beside the repo or set SERPENTGUARD_WHO_PDF"
models, thresholds = load_models(RUN_DIR, DECISION_FILE, FRONTEND_DIR, ONNX_PATH)
tables = models.tables or repo_tables()
lookup = LookupRetriever(build_kb(PDF))
decision_report = json.loads(DECISION_FILE.read_text())
print(f"{DECISION_FILE.name} checked against {RUN_DIR.name}'s own validation outputs: decide() reproduces its counts "
      f"{decision_report['verdicts']['val']['total']} = {[decision_report['verdicts']['val'][v] for v in VERDICTS]} "
      f"{list(VERDICTS)}")
print(f"run {RUN_DIR.name}: front end {models.front_end}, input {models.input_size}, backend {models.backend}, "
      f"detector {'yes' if models.detector is not None else 'none'}, tau_snake {models.tau_snake}")
print(thresholds)

decision_metrics.json checked against H1_top1_s1's own validation outputs: decide() reproduces its counts 17151 = [7778, 7737, 1636] ['dangerous', 'not_listed', 'unknown']
run H1_top1_s1: front end F1, input 320, backend torch, detector yes, tau_snake 0.102
Thresholds(temperature=1.9086784115427977, temperature_who=0.9690096288687304, theta_d=0.05660591583161018, theta_who=inf, tau_unknown=0.40982445157291975, theta_conf=0.360998147721644, theta_v=0.5, theta_genus=0.5)


## 1. The photos

The validation rows of the local sample's manifest, taken by `video.round_robin`: the classes in a seeded random
order, one photo each in turn, so 100 photos cover 100 different classes.

In [3]:
manifest = pd.read_csv(SAMPLE_DIR / "manifest.csv", dtype={"observer_id": str})
val = manifest[manifest["use"].eq("val") & manifest["split"].eq("val")]
classes = pd.read_csv("../data/classes.csv")
val = val.assign(class_id=val["species"].map(class_index(classes)))
assert val["class_id"].notna().all(), "every validation species must be in classes.csv"
photos = round_robin(val, N_PHOTOS, SEED)
photos = photos.assign(class_id=photos["class_id"].astype(int),
                       image_file=[str(SAMPLE_DIR / IMAGE_FOLDER / path) for path in photos["path"]])
dangerous = tables.dangerous[photos["class_id"].to_numpy()]
listed = tables.who[photos["class_id"].to_numpy()] > 0
print(f"{len(photos)} photos from {photos['species'].nunique()} classes: {int(dangerous.sum())} of a dangerous "
      f"species ({int(listed.sum())} WHO-listed), {int((~dangerous).sum())} not; licences "
      f"{photos['license'].value_counts().to_dict()}")

100 photos from 100 classes: 26 of a dangerous species (12 WHO-listed), 74 not; licences {'cc-by-nc': 80, 'cc-by': 14, 'cc-by-nc-sa': 5, 'cc-by-sa': 1}


## 2. P1 on 100 photos

One untimed call first, so the timed calls don't carry the first call's one-off costs. Each record holds the
photo's path, licence and attribution, the expected verdict (dangerous for a danger-table species, not listed
otherwise; "we don't know" is never expected) and any failure: `missed_danger` (a danger-table snake: WHO-listed,
viper or elapid, or keelback, not called dangerous, "we don't know" included) or `false_alarm` (a snake outside the
table called dangerous). "We don't know" on a species outside the table is the cautious answer, not a failure.

In [4]:
run_pipeline(photos["image_file"].iloc[0], models, thresholds, lookup)
results, records = {}, []
started = time.time()
for photo in photos.to_dict("records"):
    results[photo["path"]] = run_pipeline(photo["image_file"], models, thresholds, lookup)
    records.append(p1_record(photo, results[photo["path"]], tables))
print(f"{len(records)} photos in {time.time() - started:.1f} s")


def node_ms(records, unit):
    """Median ms per node; for clips, over every processed frame, with the clip's own nodes apart; and the median
    total per photo or per clip."""
    rows = [{"node": f"{entry['node']} ({CLIP_SCOPE})" if entry.get("frame") == CLIP_SCOPE else entry["node"],
             "ms": entry["ms"]} for record in records for entry in record["log"]]
    medians = pd.DataFrame(rows).groupby("node", sort=False)["ms"].median()
    return {**medians.round(3).to_dict(), f"total per {unit}": round(float(np.median([r["total_ms"] for r in records])), 3)}


photo_ms = node_ms(records, "photo")
table = pd.DataFrame(records)
print("median ms per node, photos:", photo_ms)
print(pd.crosstab(table["expected"], table["verdict"], margins=True).to_string())
print("failures:", table["failure"].value_counts().to_dict())
print("reasons for we don't know:", Counter(r for rs in table.loc[table["verdict"] == "unknown", "reason"] for r in rs))

100 photos in 9.2 s
median ms per node, photos: {'preprocess': 6.781, 'front_end': 13.922, 'snakeid': 65.986, 'decision': 0.347, 'response': 0.014, 'total per photo': 87.43}
verdict     dangerous  not_listed  unknown  All
expected                                       
dangerous          25           0        1   26
not_listed         21          40       13   74
All                46          40       14  100
failures: {'false_alarm': 21, 'missed_danger': 1}
reasons for we don't know: Counter({'unclear_between_species': 14, 'unfamiliar': 1})


### Check against the run's own saved outputs

The training run saved every validation photo's outputs on Kaggle's GPU (`val_outputs.npz`). `decision.decide` on
those rows, at the same thresholds and prototypes, should give the verdicts and top-1 species P1 gave on this CPU from
the photo files, given the same no-snake flags; a small share can differ, since Kaggle's run read the photos at its
own resolution in fp16, and the largest difference in the top-1 probability shows how close they are. The
negative control pairs each P1 photo with the saved row of the next photo, so its agreement must fall to about what
two unrelated photos share.

In [5]:
with np.load(RUN_DIR / VAL_OUTPUTS) as saved:
    row_of = {str(path): row for row, path in enumerate(saved["paths"])}
    matched = [index for index, record in enumerate(records) if record["path"] in row_of]
    rows = [row_of[records[index]["path"]] for index in matched]
    outputs = {key: saved[key][rows] for key in ("embedding", "species_logits", "who_logits", "genus_logits",
                                                    "family_logits")}
snake_flags = np.array([records[index]["snake_found"] for index in matched])
saved_decisions = decide(outputs, thresholds, models.prototypes, tables, snake_found=snake_flags)
p1_verdicts = np.array([records[index]["verdict"] for index in matched])
with_snake = [index for index in matched if records[index]["snake_found"]]
p1_top1 = np.array([records[index]["top3"][0][0] for index in with_snake])
p1_top1_prob = np.array([results[records[index]["path"]].top3[0][1] for index in with_snake])
saved_top1 = tables.species[saved_decisions.top1[snake_flags]]
saved_top1_prob = saved_decisions.top1_prob[snake_flags]
cross_check = {"photos_matched": len(matched),
               "verdict_agreement": float(np.mean(saved_decisions.label == p1_verdicts)),
               "top1_agreement": float(np.mean(saved_top1 == p1_top1)),
               "control_verdict_agreement": float(np.mean(np.roll(saved_decisions.label, 1) == p1_verdicts)),
               "control_top1_agreement": float(np.mean(np.roll(saved_top1, 1) == p1_top1)),
               "max_top1_prob_difference": float(np.abs(saved_top1_prob - p1_top1_prob).max()),
               "snake_flags_passed": int(snake_flags.sum())}
print(json.dumps(cross_check, indent=1))

{
 "photos_matched": 100,
 "verdict_agreement": 0.98,
 "top1_agreement": 0.98,
 "control_verdict_agreement": 0.37,
 "control_top1_agreement": 0.01,
 "max_top1_prob_difference": 0.18726019626040102,
 "snake_flags_passed": 100
}


## 3. P1 on 5 clips

Each clip is 10 seconds at 15 FPS from one photo as stored (`video.load_raw_rgb`, then `video.make_clip`, as V1
makes them; leaves over a central ellipse since the masks are not on this machine). The pipeline reads the frames `video.schedule` keeps at 10 a second, and the screen shows
the 4-of-6 vote. The clip frames are made in memory and dropped after each clip; none is saved.

Clip stability counts what the screen shows, and that includes the "we don't know" placeholder shown before the first
majority: a placeholder that lasts a second counts as a stable label and as a flip when the real verdict replaces
it. `first_majority_at` is when the first real verdict appears.

In [6]:
clip_records = []
started = time.time()
for index, photo in enumerate(photos.head(N_CLIPS).to_dict("records")):
    frames = make_clip(load_raw_rgb(photo["image_file"]), seed=SEED + index)
    result = run_pipeline(frames, models, thresholds, lookup)
    clip_records.append({**p1_record(photo, result, tables), "clip_seed": SEED + index})
    del frames
print(f"{len(clip_records)} clips in {time.time() - started:.1f} s")
clip_ms = node_ms(clip_records, "clip")
print("median ms per node, clip frames:", clip_ms)
clip_table = pd.DataFrame([{"species": r["species"], "expected": r["expected"], "final": r["verdict"],
                            "failure": r["failure"], **{key: r["clip"][key] for key in (
                                "processed", "effective_fps", "flips", "time_to_stable", "first_majority_at", "majority")},
                            "per_frame": r["clip"]["per_frame"]} for r in clip_records])
print(clip_table.to_string(index=False, float_format="%.2f"))

5 clips in 55.8 s
median ms per node, clip frames: {'preprocess': 4.107, 'front_end': 15.67, 'snakeid': 87.95, 'decision': 0.357, 'decision (clip)': 0.244, 'response (clip)': 0.014, 'total per clip': 10191.133}
                species   expected     final     failure  processed  effective_fps  flips  time_to_stable  first_majority_at  majority                                          per_frame
        Boiga siamensis not_listed dangerous false_alarm        101          10.00      1            0.40               0.40      True                                 {'dangerous': 101}
      Oligodon cinereus not_listed dangerous false_alarm        101          10.00     13            0.10               0.90      True  {'dangerous': 55, 'not_listed': 5, 'unknown': 41}
Sibynophis subpunctatus not_listed   unknown         NaN        101          10.00     14            0.80               0.60     False {'dangerous': 40, 'unknown': 30, 'not_listed': 31}
     Calliophis bibroni  dangerous dangerous 

## 4. Examples and saved logs

Two of each verdict (the first in run order that are not failures) and two failures: a missed danger first, the worst
one (a viper or elapid called "not listed" before any other miss), then a false alarm.

In [7]:
for folder in (OUT / "examples", TEXT_OUT / "examples"):
    folder.mkdir(parents=True, exist_ok=True)
    for old in folder.glob("*.json"):
        old.unlink()
for name, rows in (("photos.jsonl", records), ("clips.jsonl", clip_records)):
    (OUT / name).write_text("".join(json.dumps(row) + "\n" for row in rows))
examples = pick_examples(records)
by_path = photos.set_index("path")
for number, example in enumerate(examples, 1):
    name = f"{number:02d}_{example['example']}.json"
    write_json(OUT / "examples" / name, example)
    photo = {**by_path.loc[example["path"]].to_dict(), "path": example["path"]}
    write_json(TEXT_OUT / "examples" / name,
               {**p1_record(photo, results[example["path"]], tables, with_text=True), "example": example["example"]})
print(pd.DataFrame([{"example": e["example"], "species": e["species"], "expected": e["expected"],
                     "verdict": e["verdict"], "failure": e["failure"], "reason": e["reason"],
                     "top1": e["top3"][0] if e["top3"] else None, "passages": [p["id"] for p in e["passages"]]}
                    for e in examples]).to_string(index=False))

   example                 species   expected    verdict       failure                                                         reason                                  top1                                                                                            passages
 dangerous      Calliophis bibroni  dangerous  dangerous           NaN [who_listed_mass, top_species_dangerous, venomous_family_mass]          [Bungarus fasciatus, 0.4788]                  [who-110-01, who-119-04, who-118-04, who-119-03, who-120-02, who-120-05, card-023]
 dangerous      Ophiophagus hannah  dangerous  dangerous           NaN                                              [who_listed_mass]                [Ptyas mucosa, 0.5921]                  [who-110-01, who-119-04, who-118-04, who-119-03, who-120-02, who-120-05, card-008]
not_listed       Oligodon cinereus not_listed not_listed           NaN                                                             [] [Psammodynastes pulverulentus, 0.362] [note-not-li

In [8]:
summary = write_json(OUT / "summary.json", {
    "run": {"run_dir": RUN_DIR.name, "decision_file": str(DECISION_FILE), "front_end": models.front_end,
            "frontend_dir": None if FRONTEND_DIR is None else str(FRONTEND_DIR), "backend": models.backend,
            "onnx_path": None if ONNX_PATH is None else str(ONNX_PATH), "input_size": models.input_size,
            "thresholds": vars(thresholds), "tau_snake": models.tau_snake, "device": "cpu"},
    "sample": {"photos": len(photos), "classes": int(photos["species"].nunique()), "seed": SEED,
               "photos_hash": subset_hash(photos["path"]), "clips": len(clip_records),
               "clip_seeds": [r["clip_seed"] for r in clip_records]},
    "definitions": {"expected": "dangerous for a danger-table species, not listed otherwise",
                    "missed_danger": "a danger-table snake (WHO-listed, viper or elapid, keelback) not called dangerous",
                    "false_alarm": "a snake outside the danger table called dangerous",
                    "clip_stability": "counts the we-don't-know placeholder shown before the first majority"},
    "decision_file_check": decision_report["verdicts"]["val"],
    "median_ms": {"photos": photo_ms, "clip_frames": clip_ms},
    "cross_check_with_saved_outputs": cross_check,
    "verdicts": {"photos": dict(Counter(r["verdict"] for r in records)),
                 "by_expected": {expected: dict(Counter(r["verdict"] for r in records if r["expected"] == expected))
                                 for expected in ("dangerous", "not_listed")},
                 "failures": dict(Counter(r["failure"] for r in records if r["failure"])),
                 "clips": dict(Counter(r["verdict"] for r in clip_records))},
    "clips": [{"path": r["path"], "species": r["species"], "expected": r["expected"], "failure": r["failure"],
               "clip_seed": r["clip_seed"], **r["clip"]} for r in clip_records],
    "examples": [{"file": f"{n:02d}_{e['example']}.json", "path": e["path"], "example": e["example"]}
                 for n, e in enumerate(examples, 1)],
    "versions": library_versions(),
})
assert sum(summary["verdicts"]["photos"].values()) == len(photos)
print(json.dumps(summary["verdicts"], indent=1))
print("wrote", sorted(path.name for path in OUT.rglob("*.json*")))

{
 "photos": {
  "dangerous": 46,
  "not_listed": 40,
  "unknown": 14
 },
 "by_expected": {
  "dangerous": {
   "dangerous": 25,
   "unknown": 1
  },
  "not_listed": {
   "dangerous": 21,
   "not_listed": 40,
   "unknown": 13
  }
 },
 "failures": {
  "false_alarm": 21,
  "missed_danger": 1
 },
 "clips": {
  "dangerous": 4,
  "unknown": 1
 }
}
wrote ['01_dangerous.json', '02_dangerous.json', '03_not_listed.json', '04_not_listed.json', '05_unknown.json', '06_unknown.json', '07_failure.json', '08_failure.json', 'clips.jsonl', 'photos.jsonl', 'summary.json']


## Reading the result

Every number below is read from this run's results.

In [9]:
table = pd.DataFrame(records)
listed = table[table["who_listed"]]
missed = table[table["failure"] == "missed_danger"]
outside = table[table["expected"] == "not_listed"]
alarms = int((table["failure"] == "false_alarm").sum())
report_alarm = decision_report["verdict_metrics"]["false_alarm_outside_table"]
no_card = [r for r in records if r["verdict"] == "dangerous" and r["card"] is None]
clip_finals = Counter(r["verdict"] for r in clip_records)
clip_alarms = sum(r["failure"] == "false_alarm" for r in clip_records)
first = clip_records[0]["clip"]
reading = [
    f"- It agrees with the training run: on {cross_check['photos_matched']} photos, decide() on the run's saved outputs "
    f"gives the same verdict for {cross_check['verdict_agreement']:.0%} and the same top-1 species for "
    f"{cross_check['top1_agreement']:.0%}, with top-1 probabilities at most {cross_check['max_top1_prob_difference']:.1e} "
    f"apart; the shifted control agrees on {cross_check['control_verdict_agreement']:.0%} and "
    f"{cross_check['control_top1_agreement']:.0%}.",
    f"- Speed in this run: the median photo takes {photo_ms['total per photo']:.1f} ms, SnakeID "
    f"{photo_ms['snakeid']:.1f} ms of it ({models.backend}), preprocessing {photo_ms['preprocess']:.1f} ms, the "
    f"decision {photo_ms['decision']:.2f} ms and the lookup {photo_ms['response']:.2f} ms; SnakeID's median on clip "
    f"frames is {clip_ms['snakeid']:.1f} ms. Other jobs can share this laptop's CPU, so these times show where the "
    f"time goes, not the O6 latency, which V1 measures.",
    f"- WHO-listed photos called dangerous: {int((listed['verdict'] == 'dangerous').sum())} of {len(listed)}. Missed "
    f"danger, {len(missed)} photos: " + "; ".join(f"{r.species} called {r.verdict} (top-1 {r.top3[0][0]})"
                                                   for r in missed.itertuples()) + ".",
    f"- False alarms: {alarms} of the {len(outside)} photos outside the danger table ({alarms / len(outside):.2f}); "
    f"the decision report gives {report_alarm:.2f} on the whole validation split at theta_d = {thresholds.theta_d:.3f}.",
    f"- Clips: {dict(clip_finals)} at the end, {clip_alarms} of them false alarms. Each read {first['processed']} of "
    f"{first['frames']} frames ({first['skipping']}). The first real verdict appeared at "
    + ", ".join(f"{r['clip']['first_majority_at']:.1f} s" for r in clip_records)
    + "; flips and time to stable count the we-don't-know placeholder before it.",
    f"- The species card: {len(no_card)} dangerous photos show no card, because no top-3 species is in the danger "
    f"table" + (f" (first: {no_card[0]['species']}, top-1 {no_card[0]['top3'][0][0]})" if no_card else "")
    + "; a dangerous screen never shows a card that says it is not on the WHO list.",
    "- Front end: " + ("no detector, so no photo gets 'no snake found'; with FRONTEND_DIR set, a photo whose best box "
                       "is below tau_snake skips SnakeID and gets it." if models.detector is None else
                       f"{sum(not r['snake_found'] for r in records)} photos had no snake found."),
]
print("\n".join(reading))

- It agrees with the training run: on 100 photos, decide() on the run's saved outputs gives the same verdict for 98% and the same top-1 species for 98%, with top-1 probabilities at most 1.9e-01 apart; the shifted control agrees on 37% and 1%.
- Speed in this run: the median photo takes 87.4 ms, SnakeID 66.0 ms of it (torch), preprocessing 6.8 ms, the decision 0.35 ms and the lookup 0.01 ms; SnakeID's median on clip frames is 88.0 ms. Other jobs can share this laptop's CPU, so these times show where the time goes, not the O6 latency, which V1 measures.
- WHO-listed photos called dangerous: 12 of 12. Missed danger, 1 photos: Ovophis monticola called unknown (top-1 Boiga multomaculata).
- False alarms: 21 of the 74 photos outside the danger table (0.28); the decision report gives 0.30 on the whole validation split at theta_d = 0.057.
- Clips: {'dangerous': 4, 'unknown': 1} at the end, 3 of them false alarms. Each read 101 of 150 frames (the 10 frames-per-second cap). The first real verdic